In [ ]:
import gzip
import json
import random

# Configuration
input_file = ("ratings/Health_and_Household.csv.gz")
output_file = "Health_and_Household_all.csv"
sample_rate = 0.1  # 10%

def sample_dataset(input_path, output_path, rate):
    count = 0
    sampled_count = 0
    
    # Opening as 'rt' (read text) handles the decompression on-the-fly
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for line in f_in:
                count += 1
                # Randomly decide to keep this line
                if random.random() < rate:
                    f_out.write(line)
                    sampled_count += 1
                
                # Print progress every 100k lines
                if count % 100000 == 0:
                    print(f"Processed {count} lines... Saved {sampled_count}")

    print(f"Finished! Total: {count} | Sampled: {sampled_count}")

sample_dataset(input_file, output_file, sample_rate)

Read Meta

In [1]:
import pandas as pd

# Load the filter
# sample_df = pd.read_csv("samples/Health_and_Household_10_percent.csv", header=None, names=['u', 'parent_asin', 'r', 't'])
# valid_asins = set(sample_df['parent_asin'].unique())

# Use the built-in Pandas JSON parser
# chunksize keeps memory low; lines=True handles the format
print("Reading with built-in JSON parser...")
chunks = pd.read_json("meta/meta_Health_and_Household.jsonl.gz", lines=True, chunksize=1000, compression='gzip')

full_meta = []
for chunk in chunks:
    # Filter each chunk against your ASIN list
    # filtered = chunk[chunk['parent_asin'].isin(valid_asins)]
    full_meta.append(chunk)

# Combine into one final DataFrame
df_final = pd.concat(full_meta, ignore_index=True)

print(f"Success! Found {len(df_final)} matching products.")

Reading with built-in JSON parser...
Success! Found 217724 matching products.


In [2]:
print(df_final.head())
df_final = df_final.sort_values(by='rating_number', ascending=False).head(10000)

    main_category                                              title  \
0            Baby                   Chicco Viaro Travel System, Teak   
1  AMAZON FASHION  Kisbaby Four Layers Muslin Lightweight Unisex ...   
2            Baby  EZTOTZ Meals with Milton - USA Made Toddler & ...   
3            Baby                         Nuby iMonster Toddler Bowl   
4     Amazon Home  mDesign Slim Storage Organizer Container Bin w...   

   average_rating  rating_number  \
0             4.6            125   
1             5.0              2   
2             4.4             37   
3             4.4             52   
4             4.5            235   

                                            features  \
0  [Aluminum, Imported, Convenient one-hand quick...   
1  [95% Cotton, 4 Layer Muslin, Hand Wash in Cold...   
2  [WHAT IS MILTON?: Milton is the fun way for yo...   
3  [Makes feeding fun for baby and easier for par...   
4  [SMART STORAGE: This large capacity slim bin i...   

             

In [3]:
df_final.to_csv("Health_and_Household_Final.csv", sep='\x1f', index=False)

In [4]:
top_asins = df_final['parent_asin'].unique().tolist()

In [5]:
import gzip
import pandas as pd

# 1. First, get the set of target ASINs from your Top 10k Metadata
# (Assuming 'top_10k_meta' is the DataFrame we created in the previous step)
target_asins = set(df_final['parent_asin'].unique())

input_file = "ratings/Health_and_Household.csv.gz"
output_file = "Health_and_Household_Top10k_Dense.csv"

def extract_dense_reviews(input_path, output_path, target_set):
    count = 0
    saved_count = 0
    
    print(f"Starting extraction for {len(target_set)} target products...")
    
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8', newline='') as f_out:
            for line in f_in:
                count += 1
                
                # Amazon CSVs are typically: user_id, parent_asin, rating, timestamp
                # We check if the parent_asin in this line is in our target set
                # We use a simple string split to keep it fast
                parts = line.strip().split(',')
                
                if len(parts) >= 2:
                    asin = parts[1] # parent_asin is the second column
                    if asin in target_set:
                        f_out.write(line)
                        saved_count += 1
                
                if count % 1000000 == 0:
                    print(f"Processed {count/1e6:.1f}M lines... Saved {saved_count} dense reviews")

    print(f"Finished! Checked: {count} | Extracted: {saved_count}")

# Execute the targeted extraction
extract_dense_reviews(input_file, output_file, target_asins)

Starting extraction for 10000 target products...
Processed 1.0M lines... Saved 616264 dense reviews
Finished! Checked: 1241084 | Extracted: 753004


In [ ]:
df_meta = pd.read_csv("Health_and_Household_Final.csv", sep='\x1f')

In [ ]:
df_meta.head()

In [ ]:
sample_df = sample_df.sample(n=500000)

In [ ]:
import pandas as pd

def get_top_5000_asins(file_path):
    # Using a dictionary to store counts: {asin: count}
    counts = {}
    
    print("Reading metadata in chunks...")
    # Chunk size of 100,000 rows is usually safe for 8GB-16GB RAM
    reader = pd.read_csv(file_path, sep='\x1f', compression='gzip', 
                         chunksize=100000, low_memory=False)
    
    for i, chunk in enumerate(reader):
        # Assuming your meta file has 'parent_asin' and 'rating_number'
        # If 'rating_number' exists, we can use it directly
        if 'rating_number' in chunk.columns:
            for _, row in chunk[['parent_asin', 'rating_number']].iterrows():
                counts[row['parent_asin']] = row['rating_number']
        else:
            # Fallback: Count occurrences if 'rating_number' is missing
            val_counts = chunk['parent_asin'].value_counts()
            for asin, count in val_counts.items():
                counts[asin] = counts.get(asin, 0) + count
        
        if i % 10 == 0:
            print(f"Processed {i * 100000} rows...")

    # Sort by count descending and take top 5000
    top_5000 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5000]
    return [asin for asin, count in top_5000]

# Usage
meta_gz_path = "meta/meta_Health_and_Household.jsonl.gz"
top_asins = get_top_5000_asins(meta_gz_path)
print(f"Top 5 ASINs: {top_asins[:5]}")